# Loan Approval Prediction - Exploratory Data Analysis

This notebook contains comprehensive exploratory data analysis for the loan approval prediction project.

## Table of Contents
1. [Data Loading and Overview](#data-loading)
2. [Data Exploration](#data-exploration)
3. [Data Visualization](#data-visualization)
4. [Feature Engineering](#feature-engineering)
5. [Model Development](#model-development)
6. [Model Evaluation](#model-evaluation)
7. [Conclusions](#conclusions)

## 1. Data Loading and Overview {#data-loading}

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Load the dataset
df = pd.read_csv('../data/loan_prediction.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Data Exploration {#data-exploration}

In [ ]:
# Basic information about the dataset
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("\nMissing Values:")
print(df.isnull().sum())
print("\n" + "="*50)
print("\nBasic Statistics:")
print(df.describe())

In [ ]:
# Examine categorical variables
categorical_columns = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']

for col in categorical_columns:
    print(f"\n{col} - Value Counts:")
    print(df[col].value_counts())
    print("-" * 30)

## 3. Data Visualization {#data-visualization}

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(3, 3, figsize=(20, 15))
fig.suptitle('Loan Approval Dataset - Exploratory Data Analysis', fontsize=20)

# Loan Status distribution
df['Loan_Status'].value_counts().plot(kind='bar', ax=axes[0,0], color=['coral', 'lightblue'])
axes[0,0].set_title('Loan Status Distribution', fontsize=14)
axes[0,0].set_xlabel('Loan Status')
axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=0)

# Gender distribution
df['Gender'].value_counts().plot(kind='pie', ax=axes[0,1], autopct='%1.1f%%', startangle=90)
axes[0,1].set_title('Gender Distribution', fontsize=14)
axes[0,1].set_ylabel('')

# Education vs Loan Status
pd.crosstab(df['Education'], df['Loan_Status']).plot(kind='bar', ax=axes[0,2])
axes[0,2].set_title('Education vs Loan Status', fontsize=14)
axes[0,2].set_xlabel('Education')
axes[0,2].legend(['Not Approved', 'Approved'])
axes[0,2].tick_params(axis='x', rotation=45)

# Property Area distribution
df['Property_Area'].value_counts().plot(kind='bar', ax=axes[1,0], color=['gold', 'lightgreen', 'salmon'])
axes[1,0].set_title('Property Area Distribution', fontsize=14)
axes[1,0].set_xlabel('Property Area')
axes[1,0].tick_params(axis='x', rotation=0)

# Applicant Income distribution
df['ApplicantIncome'].hist(bins=30, ax=axes[1,1], color='skyblue', alpha=0.7)
axes[1,1].set_title('Applicant Income Distribution', fontsize=14)
axes[1,1].set_xlabel('Income')
axes[1,1].set_ylabel('Frequency')

# Credit History vs Loan Status
pd.crosstab(df['Credit_History'], df['Loan_Status']).plot(kind='bar', ax=axes[1,2])
axes[1,2].set_title('Credit History vs Loan Status', fontsize=14)
axes[1,2].set_xlabel('Credit History')
axes[1,2].legend(['Not Approved', 'Approved'])
axes[1,2].tick_params(axis='x', rotation=0)

# Married vs Loan Status
pd.crosstab(df['Married'], df['Loan_Status']).plot(kind='bar', ax=axes[2,0])
axes[2,0].set_title('Marital Status vs Loan Status', fontsize=14)
axes[2,0].set_xlabel('Married')
axes[2,0].legend(['Not Approved', 'Approved'])
axes[2,0].tick_params(axis='x', rotation=0)

# Loan Amount distribution
df['LoanAmount'].hist(bins=30, ax=axes[2,1], color='lightcoral', alpha=0.7)
axes[2,1].set_title('Loan Amount Distribution', fontsize=14)
axes[2,1].set_xlabel('Loan Amount')
axes[2,1].set_ylabel('Frequency')

# Self Employed vs Loan Status
pd.crosstab(df['Self_Employed'], df['Loan_Status']).plot(kind='bar', ax=axes[2,2])
axes[2,2].set_title('Self Employment vs Loan Status', fontsize=14)
axes[2,2].set_xlabel('Self Employed')
axes[2,2].legend(['Not Approved', 'Approved'])
axes[2,2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numerical variables
numerical_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('Correlation Matrix of Numerical Variables', fontsize=16)
plt.show()

In [ ]:
# Box plots to identify outliers
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Applicant Income by Loan Status
sns.boxplot(data=df, x='Loan_Status', y='ApplicantIncome', ax=axes[0,0])
axes[0,0].set_title('Applicant Income by Loan Status')

# Coapplicant Income by Loan Status
sns.boxplot(data=df, x='Loan_Status', y='CoapplicantIncome', ax=axes[0,1])
axes[0,1].set_title('Coapplicant Income by Loan Status')

# Loan Amount by Loan Status
sns.boxplot(data=df, x='Loan_Status', y='LoanAmount', ax=axes[1,0])
axes[1,0].set_title('Loan Amount by Loan Status')

# Loan Amount Term by Loan Status
sns.boxplot(data=df, x='Loan_Status', y='Loan_Amount_Term', ax=axes[1,1])
axes[1,1].set_title('Loan Amount Term by Loan Status')

plt.tight_layout()
plt.show()

## 4. Feature Engineering {#feature-engineering}

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Handle missing values
print("Handling missing values...")
print("Before handling missing values:")
print(df_processed.isnull().sum())

# Fill missing values
df_processed['Gender'].fillna(df_processed['Gender'].mode()[0], inplace=True)
df_processed['Married'].fillna(df_processed['Married'].mode()[0], inplace=True)
df_processed['Dependents'].fillna(df_processed['Dependents'].mode()[0], inplace=True)
df_processed['Self_Employed'].fillna(df_processed['Self_Employed'].mode()[0], inplace=True)
df_processed['LoanAmount'].fillna(df_processed['LoanAmount'].median(), inplace=True)
df_processed['Loan_Amount_Term'].fillna(df_processed['Loan_Amount_Term'].mode()[0], inplace=True)
df_processed['Credit_History'].fillna(df_processed['Credit_History'].mode()[0], inplace=True)

print("\nAfter handling missing values:")
print(df_processed.isnull().sum())

In [ ]:
# Feature engineering
print("Creating new features...")

# Create new features
df_processed['Total_Income'] = df_processed['ApplicantIncome'] + df_processed['CoapplicantIncome']
df_processed['Income_per_Dependent'] = df_processed['Total_Income'] / (df_processed['Dependents'].astype(str).str.replace('3+', '3').astype(int) + 1)
df_processed['Loan_Amount_per_Income'] = df_processed['LoanAmount'] / df_processed['Total_Income']

print("New features created:")
print(f"Total_Income: {df_processed['Total_Income'].describe()}")
print(f"\nIncome_per_Dependent: {df_processed['Income_per_Dependent'].describe()}")
print(f"\nLoan_Amount_per_Income: {df_processed['Loan_Amount_per_Income'].describe()}")

In [ ]:
# Visualize new features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total Income by Loan Status
sns.boxplot(data=df_processed, x='Loan_Status', y='Total_Income', ax=axes[0])
axes[0].set_title('Total Income by Loan Status')

# Income per Dependent by Loan Status
sns.boxplot(data=df_processed, x='Loan_Status', y='Income_per_Dependent', ax=axes[1])
axes[1].set_title('Income per Dependent by Loan Status')

# Loan Amount per Income by Loan Status
sns.boxplot(data=df_processed, x='Loan_Status', y='Loan_Amount_per_Income', ax=axes[2])
axes[2].set_title('Loan Amount per Income by Loan Status')

plt.tight_layout()
plt.show()

## 5. Model Development {#model-development}

In [ ]:
# Prepare data for modeling
# Encode categorical variables
label_encoders = {}
categorical_columns = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']

for column in categorical_columns:
    le = LabelEncoder()
    df_processed[column] = le.fit_transform(df_processed[column])
    label_encoders[column] = le

# Encode target variable
le_target = LabelEncoder()
df_processed['Loan_Status'] = le_target.fit_transform(df_processed['Loan_Status'])
label_encoders['Loan_Status'] = le_target

print("Label encoding completed.")
print(f"Loan Status mapping: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")

In [ ]:
# Select features and target
feature_columns = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
                  'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term',
                  'Credit_History', 'Property_Area', 'Total_Income', 'Income_per_Dependent',
                  'Loan_Amount_per_Income']

X = df_processed[feature_columns]
y = df_processed['Loan_Status']

# Handle any remaining missing values
X = X.fillna(X.median())

print(f"Final feature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nFeature columns: {feature_columns}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training target distribution: {np.bincount(y_train)}")
print(f"Test target distribution: {np.bincount(y_test)}")

In [ ]:
# Initialize models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train and evaluate models
model_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Use scaled data for Logistic Regression, original for tree-based models
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_pred_proba)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    model_results[name] = {
        'model': model,
        'accuracy': accuracy,
        'auc': auc_score,
        'cv_score': cv_mean,
        'cv_std': cv_std,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"{name} Results:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  AUC Score: {auc_score:.4f}")
    print(f"  CV Score: {cv_mean:.4f} (+/- {cv_std * 2:.4f})")

## 6. Model Evaluation {#model-evaluation}

In [ ]:
# Compare model performance
comparison_df = pd.DataFrame({
    'Model': list(model_results.keys()),
    'Accuracy': [results['accuracy'] for results in model_results.values()],
    'AUC Score': [results['auc'] for results in model_results.values()],
    'CV Score': [results['cv_score'] for results in model_results.values()],
    'CV Std': [results['cv_std'] for results in model_results.values()]
})

print("Model Comparison:")
print(comparison_df)

# Select best model based on CV score
best_model_name = comparison_df.loc[comparison_df['CV Score'].idxmax(), 'Model']
print(f"\nBest Model: {best_model_name}")

In [ ]:
# Detailed evaluation of all models
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (name, results) in enumerate(model_results.items()):
    # Confusion Matrix
    cm = confusion_matrix(y_test, results['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
               xticklabels=['Not Approved', 'Approved'],
               yticklabels=['Not Approved', 'Approved'])
    axes[idx].set_title(f'{name} - Confusion Matrix\nAccuracy: {results["accuracy"]:.3f}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

# Feature importance for tree-based models
tree_models = ['Random Forest', 'Decision Tree']
for idx, name in enumerate(tree_models):
    if name in model_results:
        model = model_results[name]['model']
        if hasattr(model, 'feature_importances_'):
            importance_df = pd.DataFrame({
                'feature': feature_columns,
                'importance': model.feature_importances_
            }).sort_values('importance', ascending=False)
            
            sns.barplot(data=importance_df.head(10), 
                       x='importance', y='feature', ax=axes[3 + idx])
            axes[3 + idx].set_title(f'{name} - Top 10 Feature Importance')
            axes[3 + idx].set_xlabel('Importance')

# Performance comparison bar plot
metrics_df = comparison_df.set_index('Model')[['Accuracy', 'AUC Score', 'CV Score']]
metrics_df.plot(kind='bar', ax=axes[5], width=0.8)
axes[5].set_title('Model Performance Comparison')
axes[5].set_ylabel('Score')
axes[5].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[5].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification reports
for name, results in model_results.items():
    print(f"\n{name} - Classification Report:")
    print("=" * 50)
    print(classification_report(y_test, results['predictions'], 
                              target_names=['Not Approved', 'Approved']))

## 7. Conclusions {#conclusions}

### Key Findings:

1. **Data Insights:**
   - The dataset contains information about loan applicants with various demographic and financial features
   - Missing values were present in several columns and were handled using appropriate imputation strategies
   - Credit History appears to be a strong predictor of loan approval

2. **Feature Engineering:**
   - Created new features like Total_Income, Income_per_Dependent, and Loan_Amount_per_Income
   - These engineered features provide additional insights into the applicant's financial situation

3. **Model Performance:**
   - Multiple machine learning algorithms were tested and compared
   - The best performing model showed good predictive capability for loan approval

### Recommendations:

1. **For Financial Institutions:**
   - Credit History is the most important factor in loan approval decisions
   - Total income and loan amount ratios are significant predictors
   - Consider implementing automated screening based on these key factors

2. **For Further Analysis:**
   - Collect more diverse data to improve model generalizability
   - Consider external economic factors that might affect loan defaults
   - Implement ensemble methods for potentially better performance

3. **Model Deployment:**
   - The model can be deployed as a screening tool for initial loan application assessment
   - Regular retraining with new data is recommended to maintain performance
   - Consider fairness and bias testing before production deployment

In [ ]:
# Example prediction function
def predict_loan_approval(applicant_data, model_name='Random Forest'):
    """
    Predict loan approval for a new applicant
    
    Parameters:
    applicant_data (dict): Dictionary containing applicant information
    model_name (str): Name of the model to use for prediction
    
    Returns:
    prediction (str): 'Approved' or 'Not Approved'
    probability (float): Probability of approval
    """
    
    # Convert input to DataFrame
    input_df = pd.DataFrame([applicant_data])
    
    # Feature engineering (same as training)
    input_df['Total_Income'] = input_df['ApplicantIncome'] + input_df['CoapplicantIncome']
    input_df['Income_per_Dependent'] = input_df['Total_Income'] / (
        input_df['Dependents'].astype(str).str.replace('3+', '3').astype(int) + 1)
    input_df['Loan_Amount_per_Income'] = input_df['LoanAmount'] / input_df['Total_Income']
    
    # Encode categorical variables
    for column in categorical_columns:
        if column in label_encoders:
            input_df[column] = label_encoders[column].transform(input_df[column])
    
    # Select features
    X_new = input_df[feature_columns]
    
    # Get model and make prediction
    model = model_results[model_name]['model']
    
    if model_name == 'Logistic Regression':
        X_new_scaled = scaler.transform(X_new)
        prediction = model.predict(X_new_scaled)[0]
        probability = model.predict_proba(X_new_scaled)[0][1]
    else:
        prediction = model.predict(X_new)[0]
        probability = model.predict_proba(X_new)[0][1]
    
    # Decode prediction
    prediction_label = label_encoders['Loan_Status'].inverse_transform([prediction])[0]
    
    return prediction_label, probability

# Example usage
sample_applicant = {
    'Gender': 'Male',
    'Married': 'Yes',
    'Dependents': '1',
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'ApplicantIncome': 5000,
    'CoapplicantIncome': 2000,
    'LoanAmount': 150,
    'Loan_Amount_Term': 360,
    'Credit_History': 1.0,
    'Property_Area': 'Urban'
}

prediction, probability = predict_loan_approval(sample_applicant, best_model_name)
print(f"\nExample Prediction using {best_model_name}:")
print(f"Prediction: {prediction}")
print(f"Approval Probability: {probability:.2%}")